In [ ]:
funnel_analysis. ipynb import pandas as pd
import numpy as np

# Load all the CSVs
orders = pd.read_csv('data/raw/orders.csv')
order_items = pd.read_csv('data/raw/order_items.csv')
payments = pd.read_csv('data/raw/order_payments.csv')
customers = pd.read_csv('data/raw/customers.csv')
products = pd.read_csv('data/raw/products.csv')
reviews = pd.read_csv('data/raw/order_reviews.csv')

# Quick peek at each
print("Orders:", orders.shape)
print("Items:", order_items.shape)
print("Payments:", payments.shape)
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Reviews:", reviews.shape)

In [ ]:
import pandas as pd
import numpy as np

orders = pd.read_csv('data/raw/orders.csv')
order_items = pd.read_csv('data/raw/order_items.csv')
payments = pd.read_csv('data/raw/order_payments.csv')
customers = pd.read_csv('data/raw/customers.csv')
products = pd.read_csv('data/raw/products.csv')
reviews = pd.read_csv('data/raw/order_reviews.csv')

print("Orders:", orders.shape)
print("Items:", order_items.shape)
print("Payments:", payments.shape)
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Reviews:", reviews.shape)

In [ ]:
import pandas as pd
import numpy as np

base = r"C:\Users\Ramsha\ecommerce-funnel-analysis\data\raw"

orders = pd.read_csv(base + r"\orders.csv")
order_items = pd.read_csv(base + r"\order_items.csv")
payments = pd.read_csv(base + r"\order_payments.csv")
customers = pd.read_csv(base + r"\customers.csv")
products = pd.read_csv(base + r"\products.csv")
reviews = pd.read_csv(base + r"\order_reviews.csv")

print("Orders:", orders.shape)
print("Items:", order_items.shape)
print("Payments:", payments.shape)
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Reviews:", reviews.shape)

In [ ]:
import os
for root, dirs, files in os.walk(r"C:\Users\Ramsha\ecommerce-funnel-analysis"):
    for f in files:
        if f.endswith('.csv'):
            print(os.path.join(root, f))

In [ ]:
import pandas as pd
import numpy as np

base = r"C:\Users\Ramsha\ecommerce-funnel-analysis\data\raw"

orders = pd.read_csv(base + r"\olist_orders_dataset.csv")
order_items = pd.read_csv(base + r"\olist_order_items_dataset.csv")
payments = pd.read_csv(base + r"\olist_order_payments_dataset.csv")
customers = pd.read_csv(base + r"\olist_customers_dataset.csv")
products = pd.read_csv(base + r"\olist_products_dataset.csv")
reviews = pd.read_csv(base + r"\olist_order_reviews_dataset.csv")

print("Orders:", orders.shape)
print("Items:", order_items.shape)
print("Payments:", payments.shape)
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Reviews:", reviews.shape)


In [ ]:
# Peek at each table's columns and a sample row
print("=== ORDERS ===")
print(orders.columns.tolist())
print(orders.head(2))
print()

print("=== ORDER ITEMS ===")
print(order_items.columns.tolist())
print(order_items.head(2))
print()

print("=== PAYMENTS ===")
print(payments.columns.tolist())
print(payments.head(2))

In [ ]:
# Load the category translation file too
translation = pd.read_csv(base + r"\product_category_name_translation.csv")

print("=== CUSTOMERS ===")
print(customers.columns.tolist())
print(customers.head(2))
print()

print("=== PRODUCTS ===")
print(products.columns.tolist())
print(products.head(2))
print()

print("=== REVIEWS ===")
print(reviews.columns.tolist())
print(reviews.head(2))
print()

print("=== TRANSLATION ===")
print(translation.head(5))

In [ ]:
# Step 2: Join everything into one analysis table

# 1. Start with orders
df = orders.copy()

# 2. Add customer location
df = df.merge(customers, on='customer_id', how='left')

# 3. Add product + price/freight via order_items
df = df.merge(order_items, on='order_id', how='left')

# 4. Add product category (and translate to English)
df = df.merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
df = df.merge(translation, on='product_category_name', how='left')

# 5. Add payment info
df = df.merge(payments, on='order_id', how='left')

# 6. Add review score
df = df.merge(reviews[['order_id', 'review_score']], on='order_id', how='left')

# Check the result
print("Merged shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
# Step 3: Clean up

# 1. Convert date columns to datetime
date_cols = ['order_purchase_timestamp', 'order_approved_at',
              'order_delivered_carrier_date', 'order_delivered_customer_date',
              'order_estimated_delivery_date']
for c in date_cols:
    df[c] = pd.to_datetime(df[c])

# 2. Check missing values
print("Missing values per column (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))
print()

#  ️ 3. Check order statuses
print("Order statuses:")
print(df['order_status'].value_counts())

In [ ]:
valid = df[df['order_status'].isin(['delivered', 'shipped', 'invoiced', 'processing', 'created', 'approved'])].copy()
valid = valid.dropna(subset=['price'])

print("Valid orders with items:", valid['order_id'].nunique())


funnel = pd.DataFrame({
    'stage': ['Orders Placed', 'Payment Approved', 'Shipped', 'Delivered'],
    'count': [
        valid['order_id'].nunique(),
        valid[valid['order_approved_at'].notna()]['order_id'].nunique(),
        valid[valid['order_delivered_carrier_date'].notna()]['order_id'].nunique(),
        valid[valid['order_delivered_customer_date'].notna()]['order_id'].nunique(),
    ]
})

funnel['conversion_from_placed'] = funnel['count'] / funnel['count'].iloc[0] * 100
funnel['dropoff_prev'] = (funnel['count'].shift(1) - funnel['count']) / funnel['count'].shift(1) * 100

print(funnel.round(1))

In [ ]:
cat_funnel = valid.groupby('product_category_name_english').agg(
    placed=('order_id', 'nunique'),
    approved=('order_approved_at', lambda x: x.notna().sum()),
    shipped=('order_delivered_carrier_date', lambda x: x.notna().sum()),
    delivered=('order_delivered_customer_date', lambda x: x.notna().sum()),
).reset_index()

cat_funnel['delivery_rate'] = cat_funnel['delivered'] / cat_funnel['placed'] * 100
cat_funnel['dropoff_placed_to_delivered'] = 100 - cat_funnel['delivery_rate']


top_cats = cat_funnel.nlargest(10, 'placed')
print(top_cats.round(1))

In [ ]:

valid_orders = valid.drop_duplicates('order_id').copy()

cat_funnel = valid_orders.groupby('product_category_name_english').agg(
    placed=('order_id', 'count'),
    approved=('order_approved_at', lambda x: x.notna().sum()),
    shipped=('order_delivered_carrier_date', lambda x: x.notna().sum()),
    delivered=('order_delivered_customer_date', lambda x: x.notna().sum()),
).reset_index()

cat_funnel['delivery_rate'] = cat_funnel['delivered'] / cat_funnel['placed'] * 100
cat_funnel['dropoff_placed_to_delivered'] = 100 - cat_funnel['delivery_rate']

top_cats = cat_funnel.nlargest(10, 'placed')
print(top_cats.round(1))

In [ ]:
# Segment funnel by customer state
state_funnel = valid_orders.groupby('customer_state').agg(
    placed=('order_id', 'count'),
    delivered=('order_delivered_customer_date', lambda x: x.notna().sum()),
).reset_index()

state_funnel['delivery_rate'] = state_funnel['delivered'] / state_funnel['placed'] * 100
state_funnel['dropoff'] = 100 - state_funnel['delivery_rate']

# Top 10 states by volume
top_states = state_funnel.nlargest(10, 'placed')
print(top_states.round(1))

In [ ]:
# Profitability by category
profit_cat = valid_orders.groupby('product_category_name_english').agg(
    revenue=('price', 'sum'),
    freight=('freight_value', 'sum'),
    orders=('order_id', 'count'),
).reset_index()

profit_cat['avg_order_value'] = profit_cat['revenue'] / profit_cat['orders']
profit_cat['freight_ratio'] = profit_cat['freight'] / profit_cat['revenue'] * 100

top_cats_profit = profit_cat.nlargest(10, 'revenue')
print(top_cats_profit.round(1))

In [ ]:
combined = cat_funnel[['product_category_name_english', 'placed', 'delivery_rate']].merge(
    profit_cat[['product_category_name_english', 'revenue', 'freight_ratio', 'avg_order_value']],
    on='product_category_name_english'
)


combined['efficiency_score'] = combined['delivery_rate'] - combined['freight_ratio']

print(combined.nlargest(10, 'revenue').round(1))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


sns.set_theme(style="whitegrid")


fig, axes = plt.subplots(1, 2, figsize=(14, 5))


sns.barplot(data=funnel, x='stage', y='count', ax=axes[0], color='#4C72B0')
axes[0].set_title('Order Funnel — Placed to Delivered')
axes[0].set_ylabel('Number of Orders')
axes[0].tick_params(axis='x', rotation=20)


top10 = combined.nlargest(10, 'revenue')
sns.barplot(data=top10, x='efficiency_score', y='product_category_name_english', ax=axes[1], color='#55A868')
axes[1].set_title('Category Efficiency Score (delivery − freight drag)')
axes[1].set_xlabel('Efficiency Score')

plt.tight_layout()
plt.show()

In [ ]:
state_chart = state_funnel.nlargest(15, 'placed')

plt.figure(figsize=(10, 6))
sns.barplot(data=state_chart, x='dropoff', y='customer_state', color='#C44E52')
plt.title('Order Drop-off Rate by State (top 15 by volume)')
plt.xlabel('Drop-off % (placed → delivered)')
plt.ylabel('State')
plt.tight_layout()
plt.show()

In [ ]:
Kernel
